In [17]:
import numpy as np
import tkinter as tk
from tkinter import filedialog
import os

def save_to_csv(data, original_file_path):
    """Handles the conversion of NumPy data into a CSV format."""
    base_path = os.path.splitext(original_file_path)[0]

    try:
        # Case 1: The .npy is a standard 1D or 2D array
        if isinstance(data, np.ndarray) and data.shape != () and data.ndim <= 2:
            csv_path = base_path + ".csv"
            # fmt="%s" ensures it writes strings, floats, and ints safely
            np.savetxt(csv_path, data, delimiter=",", fmt="%s")
            print(f"\n[+] Success: Data exported to {csv_path}")

        # Case 2: The .npy contains a dictionary
        elif data.shape == () and isinstance(data.item(), dict):
            dict_data = data.item()
            print("\n[!] Dictionary detected. Extracting arrays to CSV...")
            for key, val in dict_data.items():
                if isinstance(val, np.ndarray) and val.ndim <= 2:
                    csv_path = f"{base_path}_{key}.csv"
                    np.savetxt(csv_path, val, delimiter=",", fmt="%s")
                    print(f"  -> Saved key '{key}' to {csv_path}")
                else:
                    print(f"  -> Skipped '{key}': Contains >2D array or non-array data.")

        # Case 3: The .npy is > 2D (like images or complex tensors)
        elif isinstance(data, np.ndarray) and data.ndim > 2:
            print("\n[-] Cannot save arrays with > 2 dimensions directly to CSV.")
            print("    Hint: Flatten or reshape the array first (e.g., data.reshape(data.shape[0], -1))")

    except Exception as e:
        print(f"\n[-] Error saving to CSV: {e}")

def examine_npy_file(file_path):
    print(f"Examining file: {file_path}")
    try:
        data=np.load(file_path, allow_pickle=True)
        print(f"Shape: {data.shape}")
        print(f"Data type: {data.dtype}")
        if data.shape==() and isinstance(data.item(), dict):
            print("\nResult: Python file contains a dictionary")
            dict_data=data.item()
            print("Keys found inside:")
            for key in dict_data:
                val=dict_data[key]
                if isinstance(val, np.ndarray):
                    print(f"  - Key: '{key}' | Type: {type(val)} | Shape: {val.shape} | Data type: {val.dtype}")
                else:
                    print(f"  - Key: '{key}' | Type: {type(val)} | Value: {val}")
        elif data.dtype.names is not None:
            print("\nResult: This is a structured array")
            print("Fields Names:", data.dtype.names)
        else:
            print("\nResult: This is a standard N-dimensional array")
            if data.ndim==1 or (data.ndim==2 and data.shape[1]==1):
                unique_labels=np.unique(data)
                print(f"\nUnique labels/classes found ({len(unique_labels)} total):")
                print(unique_labels[:20])
                if len(unique_labels)>20:
                    print("... (and more)")
        save_to_csv(data, file_path)
    except Exception as e:
        print(f"\nError loading {file_path}: {e}")

if __name__=="__main__":
    root=tk.Tk()
    root.withdraw()
    print("Opening file dialog... Select .npy file")
    selected_file_path=filedialog.askopenfilename(title="Select a NumPy File", filetypes=[("NumPy files", "*.npy")])
    if selected_file_path:
        examine_npy_file(selected_file_path)
    else:                     
        print("No file selected. Exiting.")

Opening file dialog... Select .npy file
Examining file: /home/group4/Challenge3/vtp_analysis/outputs/dataset/outcomes.csv.npy
Shape: (31, 1)
Data type: int64

Result: This is a standard N-dimensional array

Unique labels/classes found (2 total):
[0 1]

[+] Success: Data exported to /home/group4/Challenge3/vtp_analysis/outputs/dataset/outcomes.csv.csv
